# KAALASTRA v2 — one bounded Nemotron risk-classification run
Attach the matching private dataset ZIP, enable a GPU and Internet, then use **Save Version → Save & Run All**.
This is a new binary classification objective. Do not mix old pilot adapters, scripts or cells with it.
Training is capped at two hours including model setup. The total 3h30 budget includes installation, downloads and evaluation.
A completed pipeline is not a guarantee of useful accuracy; inspect `candidate_gate_passed` and the actual metrics.

In [ ]:
import os, sys, time, pathlib, subprocess, json, hashlib, zipfile
SESSION_STARTED = time.time()
DEADLINE = SESSION_STARTED + 3.5 * 3600
single_gpu_env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0', 'TOKENIZERS_PARALLELISM': 'false'}
run_dir = pathlib.Path('/kaggle/working/kaalastra_run_v2')
print('Deadline (UTC):', time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(DEADLINE)))

In [ ]:
expected_zip = '77892e6b8ceaae1e85edc4104cc1e045f0201b3ee996692fa57feeba86b2dce5'
expected_files = {'train.jsonl': '1f3ae05f5842d6857e727c0e23f91d6dc1ae749c9d4cdcf1930104a79811a85b', 'validation.jsonl': '8cff6736ab5f65417f0c308c65e8b8e456a6e396cdad27a8cd3fd895256ed1cf', 'test.jsonl': 'ea676e1847c619b8378dc554292e1640f065873f00decbbe56164dd3be1d559b', 'manifest.json': '2675df3126bfd25288ca282c1378e20e51cd02ec263428429a670a3a40d3ad31', 'train_kaggle.py': 'f46d2cd2ce6cb2fb40ce32200ede7dfb6ce0fd209bed3b517d129fb3a0e09f5e', 'evaluate_kaggle.py': '8d047ff277af7b6c775a64bfbbc9ba3fea7edb1692cf31404f0a0d1f48c5fbf3', 'review_kaggle.py': '0d588fbb3a2e80a15aa24751a10e9ddb78c41cba88bcc865b3dab4ed46c3e257', 'risk_core.py': '4a8ba63778f93d01b5f02639ba139e5dc703cc4863cf98eccbf270013201d5c2', 'risk_model.py': '1a5d9f33cc3086678e74ed833a469a63b7b21b7c98106572fce88fdf8b78e576', 'audit_training_data.py': 'f4249dac059ee77b27aec84b39fd72354a0d80e2148d1857be0d841c932fc449', 'export_run.py': 'e6bac89569dcec2b64068b5a98101db9e2f6ab0200f935d888780c98dbc7cd08', 'run_kaggle.py': '10e4b3053027eeca7d7b2023dbbdd787480913269c6f461332516fb54da86b57'}
root = pathlib.Path('/kaggle/input')
matches = list(root.rglob('kaalastra-kaggle-sft.zip'))
if len(matches) == 1:
    package = matches[0]
    assert hashlib.sha256(package.read_bytes()).hexdigest() == expected_zip, 'Wrong dataset version: upload the matching ZIP.'
    data_dir = pathlib.Path('/kaggle/working/kaalastra_sft_v2')
    data_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(package) as archive:
        assert set(archive.namelist()) == set(expected_files), 'Unexpected ZIP entries'
        for entry in archive.namelist():
            assert pathlib.PurePosixPath(entry).name == entry and '\\' not in entry
        archive.extractall(data_dir)
else:
    assert not matches, 'Attach only one version of the dataset.'
    candidates = [p.parent for p in root.rglob('run_kaggle.py') if all((p.parent / name).is_file() for name in expected_files)]
    assert len(candidates) == 1, 'Attach exactly one matching v2 dataset (ZIP or unpacked files).'
    data_dir = candidates[0]
for name, expected in expected_files.items():
    actual = hashlib.sha256((data_dir / name).read_bytes()).hexdigest()
    assert actual == expected, f'{name} mismatch. Replace dataset and notebook together; do not bypass this check.'
print('Verified:', data_dir)
print(json.dumps(json.loads((data_dir / 'manifest.json').read_text())['splits'], indent=2))

The next cell uses a separate Python environment for training, inheriting Kaggle's CUDA PyTorch.
It leaves the notebook kernel's package installation unchanged. No Gradio or Diffusers imports are needed.
If the model requires authentication, add a Kaggle secret named `HF_TOKEN` before starting.

In [ ]:
venv_dir = pathlib.Path('/kaggle/working/kaalastra_env')
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(venv_dir)], check=True, timeout=120)
python = str(venv_dir / 'bin/python')
subprocess.run([sys.executable, '-m', 'pip', '--python', python, 'install', '--disable-pip-version-check',
    'transformers==4.57.6', 'peft==0.21.2', 'accelerate==1.11.0',
    'bitsandbytes==0.50.2', 'huggingface-hub==0.36.2'], check=True, timeout=900)
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token: single_gpu_env['HF_TOKEN'] = token
except Exception:
    print('No HF_TOKEN secret loaded; using public model access.')
check = "import sys; sys.path.insert(0, sys.argv[1]); import torch, transformers, peft, accelerate, bitsandbytes; from risk_model import gpu_check; print('GPU:', gpu_check().name); print('Training imports OK:', torch.__version__, transformers.__version__, peft.__version__, accelerate.__version__, bitsandbytes.__version__)"
subprocess.run([python, '-c', check, str(data_dir)], check=True, env=single_gpu_env, timeout=120)

## Train, compare, export
One command runs the corpus audit, longest-example forward/backward preflight, training, validation threshold selection, paired test and export.
There is no separate pilot. Checkpoints are saved every 40 optimizer steps; predictions are saved after each example.
If a stage fails, the run reports **incomplete** and exports whatever was saved. A timeout is not a successful evaluation.

In [ ]:
command = [python, '-u', str(data_dir / 'run_kaggle.py'), '--data-dir', str(data_dir),
    '--run-dir', str(run_dir), '--deadline-unix', str(DEADLINE)]
print('Running:', ' '.join(command))
result = subprocess.run(command, env=single_gpu_env)
print('Pipeline exit code:', result.returncode)
if result.returncode:
    print('INCOMPLETE: inspect pipeline_status.json and logs. Saved adapters and predictions are preserved.')

In [ ]:
from IPython.display import display, FileLink
for name in ('pipeline_status.json', 'heldout_results.json'):
    path = run_dir / name
    if path.exists(): print(name, path.read_text())
archive = pathlib.Path('/kaggle/working/kaalastra_adapter.zip')
if archive.exists():
    display(FileLink(str(archive)))
    print('Download kaalastra_adapter.zip from the saved notebook version Output tab.')
else:
    print('No export exists: inspect the setup cell errors.')
print('Do not reset the session until artifacts are saved or downloaded.')

## Read the result before deployment
`pipeline_status.status=complete` means the pipeline completed. `heldout_results.candidate_gate_passed` separately checks recall >= 0.60,
false-positive rate <= 0.20 and improvement in both F1 and balanced accuracy against the same base model.
This is an exploratory local comparison, not a SOTA or production certification. Prior v1 development inspected part of the test data.
The adapter classifies complete functions up to 768 prompt tokens; it does not provide verified CWE, line numbers, fixes or repository coverage.

If a session disappears, only downloaded or persisted outputs can be recovered. The ZIP includes adapter, tokenizer, manifests,
thresholds, predictions, logs and available metrics. Trainer checkpoints stay in the output directory, outside the portable ZIP.
See `docs/KAGGLE_TRAINING.md` in the project for recovery and inference commands.